# Online Shoppers Purchasing Intention — classification system evaluation

Predict whether a browsing session ends in a transaction (`Revenue`), using UCI dataset 468
(12,330 sessions, Turkish e-commerce, one year).

**Business action this model serves.** Flag likely converters in-session so live sales support
(chat, assisted checkout) is routed to them. A false negative is a lost sale; a false positive is
a wasted support slot. Lost sales cost more than wasted support, so we set
**C_FN / C_FP = 5** and test that assumption in §7.

## §0 Selection metric — declared before any model is fitted

- **Primary:** 5-fold stratified cross-validated **ROC AUC**, computed from out-of-fold
  predictions on the **training data only**.
- **Tie-break:** expected cost per case at C_FN/C_FP = 5, each model at its own
  cost-minimising threshold.
- The test set takes no part in any comparison. It is opened once, in §8, for the single
  model already chosen.

In [ ]:
import os, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis as LDA,
                                           QuadraticDiscriminantAnalysis as QDA)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

SEED, TEST_SIZE, K, COST = 42, 0.25, 5, 5.0
print(f"C_FN/C_FP = {COST}  ->  theoretical Bayes threshold t* = {1/(1+COST):.4f}")

## §1 Load and split once

The split happens before anything else, so nothing downstream can see the test set.

In [ ]:
os.makedirs("data", exist_ok=True)
p = "data/uci_468.csv"
if not os.path.exists(p):
    r = requests.get("https://archive.ics.uci.edu/static/public/468/data.csv", timeout=600)
    r.raise_for_status(); open(p, "wb").write(r.content)

df = pd.read_csv(p)
y_all = df.pop("Revenue").astype(bool).astype(int)
# Integer-coded nominal IDs and the boolean flag are categories, not quantities:
# cast to string so they are one-hot encoded and never scaled.
for c in ["OperatingSystems", "Browser", "Region", "TrafficType", "Weekend"]:
    df[c] = df[c].astype(str)

Xtr, Xte, ytr, yte = train_test_split(df, y_all, test_size=TEST_SIZE,
                                      stratify=y_all, random_state=SEED)
Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
ytr, yte = ytr.reset_index(drop=True), yte.reset_index(drop=True)
del df, y_all                      # remove the pooled frame so it cannot be used by accident

print(f"train {len(Xtr):,} rows, {int(ytr.sum()):,} positives ({ytr.mean():.4f})")
print(f"test  {len(Xte):,} rows, {int(yte.sum()):,} positives ({yte.mean():.4f})   [SEALED until §8]")

## §2 EDA — training data only

In [ ]:
def split_types(X, max_card=12):
    """Numeric if continuous. A numeric column with few levels is really categorical."""
    num, cat = [], []
    for c in X.columns:
        s = X[c]
        if pd.api.types.is_numeric_dtype(s):
            (cat if s.nunique(dropna=True) <= max_card else num).append(c)
        else:
            cat.append(c)
    return num, cat

NUM, CAT = split_types(Xtr)
display(pd.DataFrame({"dtype": Xtr.dtypes.astype(str),
                      "n_unique": Xtr.nunique(),
                      "pct_missing": Xtr.isna().mean().round(4),
                      "role": ["numeric" if c in NUM else "categorical" for c in Xtr.columns]}))
print(f"\n{len(NUM)} numeric, {len(CAT)} categorical | class balance: "
      f"{int(ytr.sum()):,} convert vs {int((1-ytr).sum()):,} do not")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].bar(["no purchase", "purchase"], ytr.value_counts().sort_index().values, color=["#9fb3c8", "#c0562c"])
ax[0].set_title("Class balance (train)"); ax[0].set_ylabel("sessions")

Xtr[NUM].plot(kind="box", ax=ax[1], vert=False, logy=False)
ax[1].set_xscale("symlog"); ax[1].set_title("Numeric features (symlog)"); ax[1].tick_params(labelsize=7)

Cm = Xtr[NUM].corr().abs().values
im = ax[2].imshow(Cm, vmin=0, vmax=1, cmap="magma")
ax[2].set_xticks(range(len(NUM))); ax[2].set_xticklabels(NUM, rotation=90, fontsize=6)
ax[2].set_yticks(range(len(NUM))); ax[2].set_yticklabels(NUM, fontsize=6)
ax[2].set_title("|correlation|, numeric features"); fig.colorbar(im, ax=ax[2], shrink=.8)
plt.tight_layout(); plt.show()

off = Cm[~np.eye(len(NUM), dtype=bool)]
print(f"mean |corr| {off.mean():.3f} | 95th pct {np.percentile(off,95):.3f} | max {off.max():.3f}")
print("Bearing: Naive Bayes assumes conditional independence. The higher these are, the more")
print("NB should underperform Logistic/LDA, which model the covariance.")

## §3 Leakage audit — training data only

A univariate screen ranks each feature by how well it separates the classes on its own.
A high score is not proof of leakage: a genuinely strong predictor scores high too. It only
tells us where to look.

In [ ]:
def make_pre(num, cat):
    """Every transform lives here, so the pipeline refits it on each training fold."""
    parts = []
    if num: parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                           ("sc",  StandardScaler())]), num))
    if cat: parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                           ("oh",  OneHotEncoder(drop="if_binary",
                                                   handle_unknown="infrequent_if_exist",
                                                   min_frequency=0.005, max_categories=30,
                                                   sparse_output=False))]), cat))
    return ColumnTransformer(parts, remainder="drop")

skf = StratifiedKFold(K, shuffle=True, random_state=SEED)

def cv_oof(X, mk):
    """Out-of-fold probabilities. Preprocessing is refitted inside every fold."""
    pipe = Pipeline([("pre", make_pre(*split_types(X))), ("clf", mk())])
    return cross_val_predict(pipe, X, ytr, cv=skf, method="predict_proba")[:, 1]

rows = []
for c in Xtr.columns:
    if c in NUM:
        v = pd.to_numeric(Xtr[c], errors="coerce").fillna(Xtr[c].median())
        a = roc_auc_score(ytr, v); a = max(a, 1 - a)
    else:
        a = roc_auc_score(ytr, cv_oof(Xtr[[c]], lambda: LogisticRegression(max_iter=1000)))
    rows.append({"feature": c, "univariate_AUC": a})
screen = pd.DataFrame(rows).sort_values("univariate_AUC", ascending=False).reset_index(drop=True)
display(screen)

**The `PageValues` decision.** `PageValues` is a Google Analytics figure: the average value of
the pages a user visited, where that value is assigned from sessions that **completed a
transaction**. It is therefore computed partly from the outcome we are predicting, and a model
using it would not be reproducible at prediction time.

We measure it rather than assert it — CV AUC with it against CV AUC without, both on training
data. The difference is the **leakage premium**: performance that would not survive deployment.

In [ ]:
lr = lambda: LogisticRegression(max_iter=3000)
auc_with    = roc_auc_score(ytr, cv_oof(Xtr, lr))
auc_without = roc_auc_score(ytr, cv_oof(Xtr.drop(columns=["PageValues"]), lr))
print(f"CV AUC including PageValues : {auc_with:.4f}")
print(f"CV AUC excluding PageValues : {auc_without:.4f}")
print(f"leakage premium             : {auc_with - auc_without:.4f}")
print("\nDECISION: drop PageValues. The premium above is the cost of honesty; every number")
print("from here on excludes it, so nothing downstream is inflated by outcome-derived data.")

Xtr = Xtr.drop(columns=["PageValues"])
Xte = Xte.drop(columns=["PageValues"])
NUM, CAT = split_types(Xtr)
print(f"\nfeatures now: {len(NUM)} numeric, {len(CAT)} categorical")

## §4 Leakage-free pipeline

Imputation, scaling and one-hot encoding are steps **inside** the `Pipeline`, never applied to
the training set beforehand. `cross_val_predict` therefore refits them on each fold's training
part only, so no fold sees statistics derived from its own validation rows.

In [ ]:
demo = Pipeline([("pre", make_pre(NUM, CAT)), ("clf", LogisticRegression(max_iter=3000))])
display(demo)

## §5 Three models under 5-fold CV

All figures below come from out-of-fold predictions on the training data. Confusion matrices
are reported at the default threshold 0.5. Sensitivity there is poor for every model: only 15%
of sessions convert, so 0.5 is a badly mismatched default. §7 replaces it with a justified
operating point — that is the point of the extension, not a defect in the models.

In [ ]:
MODELS = {"Logistic":   lambda: LogisticRegression(max_iter=3000),
          "GaussianNB": lambda: GaussianNB(),
          "LDA":        lambda: LDA(solver="svd")}

def rates(y, p, t):
    tn, fp, fn, tp = confusion_matrix(y, (np.asarray(p) >= t).astype(int), labels=[0, 1]).ravel()
    return dict(TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan)

OOF, tab = {}, []
for m, mk in MODELS.items():
    p = cv_oof(Xtr, mk); OOF[m] = p
    folds = np.array([roc_auc_score(ytr.iloc[v], p[v]) for _, v in skf.split(Xtr, ytr)])
    tab.append({"model": m, "CV AUC": roc_auc_score(ytr, p), "fold SD": folds.std(ddof=1),
                "Brier": brier_score_loss(ytr, p), **rates(ytr, p, 0.5)})
CV = pd.DataFrame(tab).set_index("model")
display(CV.round(4))

In [ ]:
plt.figure(figsize=(5.6, 5.2))
for m in MODELS:
    fpr, tpr, _ = roc_curve(ytr, OOF[m])
    plt.plot(fpr, tpr, lw=1.8, label=f"{m}  (AUC {CV.loc[m,'CV AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.9, label="chance")
plt.xlabel("1 - specificity  (false positive rate)")
plt.ylabel("sensitivity  (true positive rate)")
plt.title("ROC — 5-fold cross-validated, out-of-fold predictions\nOnline Shoppers Purchasing Intention (training data)")
plt.legend(loc="lower right"); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

## §6 Choosing the final model

Two grounds, as required: predictive performance, then the relative cost of the two error types.
Each is compared against an alternative rather than asserted.

In [ ]:
# --- Ground 1: performance. Paired per-fold differences, so the comparison is within-fold.
fold = {m: np.array([roc_auc_score(ytr.iloc[v], OOF[m][v]) for _, v in skf.split(Xtr, ytr)])
        for m in MODELS}
best_auc  = CV["CV AUC"].idxmax(); worst = CV["CV AUC"].idxmin()
d  = fold[best_auc] - fold[worst]; se = d.std(ddof=1)/np.sqrt(len(d))
print(f"Ground 1 — best by CV AUC: {best_auc}")
print(f"  {best_auc} vs {worst}: mean paired fold gap {d.mean():+.4f}, SE {se:.4f}, "
      f"separated beyond noise: {abs(d.mean()) > 2*se}")

# --- Ground 2: cost. Each model judged at ITS OWN cost-minimising threshold.
def cost_curve(y, p, ratio=COST):
    ts = np.linspace(0.005, 0.995, 199); y = np.asarray(y)
    P, N = (y == 1).sum(), (y == 0).sum()
    c, se_, sp_ = [], [], []
    for t in ts:
        pr = (p >= t).astype(int)
        tp = ((pr == 1) & (y == 1)).sum(); fp = ((pr == 1) & (y == 0)).sum()
        c.append(((P - tp)*ratio + fp)/len(y)); se_.append(tp/P); sp_.append((N - fp)/N)
    return ts, np.array(c), np.array(se_), np.array(sp_)

cost_tab = []
for m in MODELS:
    ts, c, _, _ = cost_curve(ytr, OOF[m])
    cost_tab.append({"model": m, "own best threshold": ts[c.argmin()],
                     "expected cost/case": c.min(), "cost at t=0.5": c[np.abs(ts-0.5).argmin()]})
COSTS = pd.DataFrame(cost_tab).set_index("model")
display(COSTS.round(4))
best_cost = COSTS["expected cost/case"].idxmin()
gap = COSTS["expected cost/case"].sort_values()
print(f"Ground 2 — best by expected cost: {best_cost} "
      f"({(gap.iloc[1]-gap.iloc[0])/gap.iloc[0]*100:.2f}% cheaper than next)")

In [ ]:
# --- Depth: alternatives to the three required models, all judged by the same CV metric.
ALT = {"Logistic + class_weight": lambda: LogisticRegression(max_iter=3000, class_weight="balanced"),
       "LDA + shrinkage":         lambda: LDA(solver="lsqr", shrinkage="auto"),
       "QDA (drops equal-cov)":   lambda: QDA(reg_param=0.05)}
alt = [{"variant": k, "CV AUC": roc_auc_score(ytr, cv_oof(Xtr, mk))} for k, mk in ALT.items()]
A = pd.DataFrame(alt).set_index("variant"); display(A.round(4))
print(f"Best required model {best_auc}: {CV.loc[best_auc,'CV AUC']:.4f}")
print(f"  class_weight        {A.loc['Logistic + class_weight','CV AUC']-CV.loc['Logistic','CV AUC']:+.4f} vs plain Logistic  (re-weighting shifts calibration, not ranking)")
print(f"  LDA shrinkage       {A.loc['LDA + shrinkage','CV AUC']-CV.loc['LDA','CV AUC']:+.4f} vs plain LDA         (justifies keeping the simpler solver)")
print(f"  QDA - LDA           {A.loc['QDA (drops equal-cov)','CV AUC']-CV.loc['LDA','CV AUC']:+.4f}                       (prices LDA's equal-covariance assumption)")

In [ ]:
FINAL = best_auc if best_auc == best_cost else best_cost
print(f"Ground 1 (CV AUC)      -> {best_auc}")
print(f"Ground 2 (expected cost) -> {best_cost}")
print(f"\nFINAL MODEL: {FINAL}" + ("  (both grounds agree)" if best_auc == best_cost
      else "  (grounds disagree; cost decides, since it encodes the business objective)"))

## §7 Threshold analysis

0.5 is only optimal when the two errors cost the same. They do not: a missed converter costs
five wasted support slots. The Bayes-optimal threshold under that ratio is
t\* = 1/(1+5) = 0.167. We choose the operating point empirically, by minimising expected cost
on **training out-of-fold predictions**, and check it against t\*.

In [ ]:
ts, cost, sens, spec = cost_curve(ytr, OOF[FINAL])
T_EMP  = float(ts[cost.argmin()]); T_THEO = 1/(1+COST)
prev   = float(ytr.mean()); trivial = min(prev*COST, 1-prev)

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax[0].plot(ts, sens, lw=1.8, label="sensitivity")
ax[0].plot(ts, spec, lw=1.8, ls="--", label="specificity")
ax[0].axvline(T_EMP, color="k", ls=":", lw=1.4, label=f"chosen t = {T_EMP:.3f}")
ax[0].axvline(0.5, color="grey", ls=":", lw=1.2, label="default 0.5")
ax[0].set_xlabel("classification threshold"); ax[0].set_ylabel("rate")
ax[0].set_title(f"Sensitivity / specificity vs threshold — {FINAL}"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.25)

ax[1].plot(ts, cost, lw=1.8, color="#c0562c", label="expected cost per case")
ax[1].plot(T_EMP, cost.min(), "o", ms=7, color="k", label=f"minimum at t={T_EMP:.3f}")
ax[1].axvline(T_THEO, color="b", ls=":", lw=1.3, label=f"theoretical t* = {T_THEO:.3f}")
ax[1].axhline(trivial, color="r", ls="--", lw=1.2, label=f"best trivial rule = {trivial:.3f}")
ax[1].set_xlabel("classification threshold"); ax[1].set_ylabel(f"expected cost (C_FN/C_FP = {COST})")
ax[1].set_title("Expected cost vs threshold"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.25)
plt.tight_layout(); plt.show()

c_half = cost[np.abs(ts-0.5).argmin()]
print(f"cost at 0.5 {c_half:.4f} | at t* {cost[np.abs(ts-T_THEO).argmin()]:.4f} | at chosen {cost.min():.4f}")
print(f"chosen threshold cuts cost {100*(c_half-cost.min())/c_half:.1f}% vs 0.5, "
      f"{100*(trivial-cost.min())/trivial:.1f}% vs the best trivial rule")
print(f"CV sensitivity/specificity at t={T_EMP:.3f}: "
      f"{sens[cost.argmin()]:.3f} / {spec[cost.argmin()]:.3f}")

In [ ]:
# Is the choice robust to the cost ratio we assumed? Re-derive it for other plausible ratios.
sw = []
for r in [2, 3, 5, 8, 10]:
    t2, c2, s2, sp2 = cost_curve(ytr, OOF[FINAL], ratio=r)
    i = c2.argmin()
    sw.append({"C_FN/C_FP": r, "theoretical t*": 1/(1+r), "empirical t": t2[i],
               "CV sensitivity": s2[i], "CV specificity": sp2[i]})
display(pd.DataFrame(sw).set_index("C_FN/C_FP").round(4))
print("The empirical threshold tracks the theoretical one across the range, so the operating")
print("point follows the cost assumption rather than an artefact of this particular ratio.")

## §8 Held-out audit — one model, one look

The test set has not been touched. It is opened now, once, for the single model chosen in §6 at
the threshold fixed in §7. This audits generalisation; it is not a chance to reconsider.

In [ ]:
final_fit = Pipeline([("pre", make_pre(NUM, CAT)), ("clf", MODELS[FINAL]())]).fit(Xtr, ytr)
p_test = final_fit.predict_proba(Xte)[:, 1]          # <-- the only use of the test set

r = rates(yte, p_test, T_EMP)
test_auc = roc_auc_score(yte, p_test)

def wilson(k, n):
    if not n: return (np.nan, np.nan)
    z = 1.959963985; ph = k/n; d = 1 + z*z/n
    c = (ph + z*z/(2*n))/d; h = z*np.sqrt(ph*(1-ph)/n + z*z/(4*n*n))/d
    return max(0., c-h), min(1., c+h)

lo, hi = wilson(r["TP"], r["TP"]+r["FN"])
slo, shi = wilson(r["TN"], r["TN"]+r["FP"])

print(f"FINAL MODEL: {FINAL}   |   operating threshold t = {T_EMP:.3f}\n")
display(pd.DataFrame([[r["TN"], r["FP"]], [r["FN"], r["TP"]]],
                     index=["actual: no purchase", "actual: purchase"],
                     columns=["predicted: no", "predicted: yes"]))
print(f"\nsensitivity {r['sensitivity']:.4f}  95% CI [{lo:.3f}, {hi:.3f}]")
print(f"specificity {r['specificity']:.4f}  95% CI [{slo:.3f}, {shi:.3f}]")
print(f"test AUC    {test_auc:.4f}")

cvv = CV.loc[FINAL, "CV AUC"]
display(pd.DataFrame([{"estimate": "CV (5-fold, train)", "AUC": cvv,
                       "sensitivity": sens[cost.argmin()], "specificity": spec[cost.argmin()]},
                      {"estimate": "held-out (once)", "AUC": test_auc,
                       "sensitivity": r["sensitivity"], "specificity": r["specificity"]}]).set_index("estimate").round(4))
print(f"CV minus held-out AUC: {cvv - test_auc:+.4f}")

## §9 Refit on the full data

With evaluation finished, the selected specification is refitted on train + test combined. This
is the model to deploy; its performance estimate remains the held-out figure from §8, not
anything measured here.

In [ ]:
X_all = pd.concat([Xtr, Xte], ignore_index=True)
y_all = pd.concat([ytr, yte], ignore_index=True)
prod = Pipeline([("pre", make_pre(NUM, CAT)), ("clf", MODELS[FINAL]())]).fit(X_all, y_all)
print(f"refitted {FINAL} on all {len(X_all):,} rows")

clf = prod.named_steps["clf"]
names = prod.named_steps["pre"].get_feature_names_out()
coef = getattr(clf, "coef_", None)
if coef is not None:
    co = (pd.DataFrame({"feature": names, "coefficient": coef[0]})
            .assign(abs_coef=lambda d: d.coefficient.abs())
            .sort_values("abs_coef", ascending=False).drop(columns="abs_coef").head(15))
    print("\nTop 15 coefficients (standardised inputs, log-odds):")
    display(co.reset_index(drop=True).round(4))

## §10 Conclusion

In [ ]:
print(f"""
Final model      : {FINAL}, operating at threshold {T_EMP:.3f} rather than 0.5
Chosen because   : highest 5-fold CV AUC ({CV.loc[FINAL,'CV AUC']:.4f}) and lowest expected
                   cost per case ({COSTS.loc[FINAL,'expected cost/case']:.4f}) at C_FN/C_FP = {COST:.0f}
Held-out audit   : AUC {test_auc:.4f}, sensitivity {r['sensitivity']:.3f}, specificity {r['specificity']:.3f}
Generalisation   : CV minus held-out AUC = {cvv - test_auc:+.4f}
Leakage handled  : PageValues removed; it was worth {auc_with - auc_without:.4f} AUC of unusable signal

Limitations
  - One year, one retailer: seasonality beyond the observed months is untested.
  - The cost ratio 5 is a modelling assumption, not a measured figure; §7 shows the operating
    point moves with it, so a revised ratio changes the threshold but not the model choice.
  - Sessions are treated as independent; repeat visitors are not linked across sessions.
""")